# Qwen2 Model Distillation Notebook

This notebook distills knowledge from a teacher model into three Qwen2 student models:
1. **Qwen2-0.5B** (smallest, trained first)
2. **Qwen2-1.5B** (medium)
3. **Qwen2-7B** (largest, trained last)

> **Note:** There is no official Qwen2-4B model. If you need a 4B model, consider using Qwen2.5-3B instead.

This notebook reuses functions from `distil_logits.py` for consistency and maintainability.

## 1. Imports and Environment Setup

In [ ]:
# Check and install/upgrade required dependencies
# Run this cell first if you encounter version-related errors

import subprocess
import sys

def check_and_upgrade_deps():
    """Check package versions and upgrade if needed."""
    required_versions = {
        "huggingface_hub": "0.21.0",  # Minimum version for HF_HUB_ENABLE_HF_TRANSFER
        "transformers": "4.40.0",
        "datasets": "2.18.0",
        "trl": "0.8.0",
    }
    
    import importlib.metadata
    
    needs_upgrade = []
    for package, min_version in required_versions.items():
        try:
            installed = importlib.metadata.version(package)
            from packaging import version
            if version.parse(installed) < version.parse(min_version):
                needs_upgrade.append(f"{package}>={min_version}")
                print(f"  {package}: {installed} -> needs >={min_version}")
            else:
                print(f"  {package}: {installed} (OK)")
        except importlib.metadata.PackageNotFoundError:
            needs_upgrade.append(f"{package}>={min_version}")
            print(f"  {package}: NOT INSTALLED")
    
    return needs_upgrade

print("Checking package versions...")
upgrades_needed = check_and_upgrade_deps()

if upgrades_needed:
    print(f"\nPackages need upgrading: {upgrades_needed}")
    print("Run the following command to upgrade:")
    print(f"  pip install --upgrade {' '.join(upgrades_needed)}")
    print("\nOr uncomment and run the cell below to auto-upgrade.")
else:
    print("\nAll package versions are compatible!")

In [ ]:
# Uncomment and run this cell to auto-upgrade packages if needed
# After upgrading, restart the kernel and run from the beginning

# !pip install --upgrade huggingface_hub>=0.21.0 transformers>=4.40.0 datasets>=2.18.0 trl>=0.8.0

In [ ]:
import os
import sys
import importlib.util
import torch
import torch.nn.functional as F
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
import yaml
import gc

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Check Flash Attention availability
FLASH_ATTN_AVAILABLE = importlib.util.find_spec("flash_attn") is not None
print(f"Flash Attention 2 available: {FLASH_ATTN_AVAILABLE}")
if not FLASH_ATTN_AVAILABLE:
    print("  -> Will use default attention implementation")

## 2. Reusable Functions from distil_logits.py

These functions are imported/adapted from the original `distil_logits.py` script.

In [ ]:
def sharegpt_format(example, tokenizer, chat_template):
    """
    Convert ShareGPT format conversations to chat format.
    Adapted from distil_logits.py
    """
    conversations = example['conversations']
    message = []
    
    if isinstance(conversations, list):
        for conversation in conversations:
            if isinstance(conversation, dict):
                if conversation.get('from') == 'human':
                    message.append({"role": "user", "content": conversation.get('value', '')})
                elif conversation.get('from') == 'gpt':
                    message.append({"role": "assistant", "content": conversation.get('value', '')})
                elif conversation.get('from') == 'system':
                    message.insert(0, {"role": "system", "content": conversation.get('value', '')})

    if not any(msg.get('role') == 'system' for msg in message):
        message.insert(0, {"role": "system", "content": "You are a helpful assistant."})

    # Apply chat template
    tokenizer.chat_template = chat_template
    text = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    return {"text": text}


def tokenize_function(examples, tokenizer, max_length):
    """
    Tokenize examples with truncation and padding.
    Adapted from distil_logits.py
    """
    return tokenizer(examples["text"], truncation=True, max_length=max_length, padding="max_length")


def pad_logits(student_logits, teacher_logits):
    """
    Pad logits to match vocabulary sizes between student and teacher models.
    Directly from distil_logits.py
    """
    student_size, teacher_size = student_logits.size(-1), teacher_logits.size(-1)
    if student_size != teacher_size:
        pad_size = abs(student_size - teacher_size)
        pad_tensor = torch.zeros((*teacher_logits.shape[:-1], pad_size), dtype=teacher_logits.dtype, device=teacher_logits.device)
        return (torch.cat([student_logits, pad_tensor], dim=-1), teacher_logits) if student_size < teacher_size else (student_logits, torch.cat([teacher_logits, pad_tensor], dim=-1))
    return student_logits, teacher_logits


def freeze_student_spectrum(model, unfrozen_layers_file):
    """
    Freeze layers of the student model based on spectrum configuration.
    Directly from distil_logits.py
    """
    with open(unfrozen_layers_file, 'r') as file:
        unfrozen_layers = yaml.safe_load(file)['unfrozen_parameters']
    
    for name, param in model.named_parameters():
        if not any(layer in name for layer in unfrozen_layers):
            param.requires_grad = False
        else:
            param.requires_grad = True

In [ ]:
class LogitsTrainer(SFTTrainer):
    """
    Custom trainer for logit-based knowledge distillation.
    Directly from distil_logits.py with parameterized distillation settings.
    """
    def __init__(self, *args, temperature=2.0, alpha=0.5, max_length=4096, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature
        self.alpha = alpha
        self.max_length = max_length
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        device = next(model.parameters()).device
        inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}
        self.teacher_model = self.teacher_model.to(device)
        
        student_model = model.module if hasattr(model, 'module') else model
        teacher_model = self.teacher_model.module if hasattr(self.teacher_model, 'module') else self.teacher_model

        student_outputs = student_model(**inputs)
        with torch.no_grad():
            teacher_outputs = teacher_model(**inputs)

        custom_loss = self.distillation_loss(model, student_outputs.logits, teacher_outputs.logits, inputs, student_outputs.loss)
        return (custom_loss, student_outputs) if return_outputs else custom_loss

    def distillation_loss(self, model, student_logits, teacher_logits, inputs, original_loss):
        device = next(model.parameters()).device
        student_logits, teacher_logits = pad_logits(student_logits.to(device), teacher_logits.to(device))
        
        student_logits_scaled = student_logits / self.temperature
        teacher_logits_scaled = teacher_logits / self.temperature

        loss_kd = F.kl_div(
            F.log_softmax(student_logits_scaled, dim=-1),
            F.softmax(teacher_logits_scaled, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2) / self.max_length

        return self.alpha * loss_kd + (1 - self.alpha) * original_loss

## 3. Helper Functions for Multi-Model Distillation

In [ ]:
def load_and_prepare_dataset(dataset_config, tokenizer, chat_template, max_length, num_proc=8):
    """
    Load and prepare dataset for training.
    """
    print(f"Loading dataset: {dataset_config['name']}")
    dataset = load_dataset(dataset_config["name"], split=dataset_config["split"])
    dataset = dataset.shuffle(seed=dataset_config["seed"])
    
    if "num_samples" in dataset_config and dataset_config["num_samples"]:
        dataset = dataset.select(range(dataset_config["num_samples"]))
        print(f"Using {dataset_config['num_samples']} samples")
    
    # Preprocess dataset
    print("Preprocessing and tokenizing dataset...")
    original_columns = dataset.column_names
    
    # Apply sharegpt format
    dataset = dataset.map(
        lambda x: sharegpt_format(x, tokenizer, chat_template),
        remove_columns=original_columns
    )
    
    # Tokenize
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=True,
        num_proc=num_proc,
        remove_columns=["text"]
    )
    
    # Split into train/test
    tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)
    print(f"Dataset prepared: {len(tokenized_dataset['train'])} train, {len(tokenized_dataset['test'])} test samples")
    
    return tokenized_dataset


def load_model(model_name, use_flash_attention=False):
    """
    Load a model with optional flash attention.
    Falls back to default attention if flash_attn is not available.
    """
    print(f"Loading model: {model_name}")
    model_kwargs = {"torch_dtype": torch.bfloat16}
    
    if use_flash_attention:
        if FLASH_ATTN_AVAILABLE:
            model_kwargs["attn_implementation"] = "flash_attention_2"
            print("  Using Flash Attention 2")
        else:
            print("  Flash Attention 2 requested but not available, using default attention")
    else:
        print("  Using default attention implementation")
    
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    print(f"Model loaded: {model_name}")
    return model


def cleanup_memory():
    """
    Clean up GPU memory between training runs.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("Memory cleaned up")

In [ ]:
def train_student_model(student_model_name, teacher_model, tokenized_dataset, config):
    """
    Train a single student model using knowledge distillation.
    """
    print(f"\n{'='*60}")
    print(f"Training Student Model: {student_model_name}")
    print(f"{'='*60}\n")
    
    # Load student model
    student_model = load_model(
        student_model_name,
        use_flash_attention=config["model_config"]["use_flash_attention"]
    )
    
    # Apply spectrum freezing if configured
    if "spectrum" in config and "layers_to_unfreeze" in config.get("spectrum", {}):
        print("Applying Spectrum layer freezing...")
        freeze_student_spectrum(student_model, config["spectrum"]["layers_to_unfreeze"])
    else:
        print("All layers of the student model will be trainable.")
    
    # Set up output directory for this model
    model_short_name = student_model_name.split('/')[-1]
    output_dir = os.path.join(config["training"]["output_dir"], model_short_name)
    
    # Update training arguments for this model
    training_config = config["training"].copy()
    training_config["output_dir"] = output_dir
    training_arguments = TrainingArguments(**training_config)
    
    # Create trainer
    trainer = LogitsTrainer(
        model=student_model,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        args=training_arguments,
        temperature=config["distillation"]["temperature"],
        alpha=config["distillation"]["alpha"],
        max_length=config["tokenizer"]["max_length"]
    )
    
    # Add teacher model to trainer
    trainer.teacher_model = teacher_model
    
    # Train
    print(f"Starting training for {student_model_name}...")
    trainer.train(resume_from_checkpoint=config["training"]["resume_from_checkpoint"])
    
    # Save the final model
    trainer.save_model(output_dir)
    print(f"Model saved to: {output_dir}")
    
    # Cleanup
    del student_model
    del trainer
    cleanup_memory()
    
    return output_dir

## 4. Configuration

Define the configuration for distillation. Modify these settings as needed.

In [ ]:
# Main configuration
config = {
    "project_name": "distil-qwen2-multi",
    "dataset": {
        "name": "mlabonne/FineTome-100k",
        "split": "train",
        "num_samples": 10000,  # Limit samples for faster training; set to None for full dataset
        "seed": 42
    },
    "models": {
        "teacher": "arcee-ai/Arcee-Spark",
        # Student models ordered from smallest to largest
        "students": [
            "Qwen/Qwen2-0.5B",   # Smallest - trained first
            "Qwen/Qwen2-1.5B",   # Medium
            "Qwen/Qwen2-7B"      # Largest - trained last (no 4B exists)
        ]
    },
    "tokenizer": {
        "max_length": 4096,
        "chat_template": "{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}{% endif %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    },
    "training": {
        "output_dir": "./results",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "save_steps": 1000,
        "logging_steps": 10,
        "learning_rate": 2e-5,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "resume_from_checkpoint": None,
        "fp16": False,
        "bf16": True
    },
    "distillation": {
        "temperature": 2.0,
        "alpha": 0.5
    },
    "model_config": {
        # Auto-detect Flash Attention 2 availability (set to True to force if installed)
        "use_flash_attention": FLASH_ATTN_AVAILABLE
    }
    # Uncomment to use Spectrum layer freezing:
    # "spectrum": {
    #     "layers_to_unfreeze": "/path/to/spectrum_config.yaml"
    # }
}

# Set wandb project
os.environ['WANDB_PROJECT'] = config["project_name"]

print("Configuration loaded:")
print(f"  Teacher model: {config['models']['teacher']}")
print(f"  Student models: {config['models']['students']}")
print(f"  Dataset: {config['dataset']['name']}")
print(f"  Max length: {config['tokenizer']['max_length']}")
print(f"  Flash Attention 2: {config['model_config']['use_flash_attention']}")

## 5. Load Teacher Model and Prepare Dataset

Load the teacher model once and reuse it for all student models.

In [ ]:
# Load teacher model (shared across all distillations)
print("Loading teacher model...")
teacher_model = load_model(
    config["models"]["teacher"],
    use_flash_attention=config["model_config"]["use_flash_attention"]
)
teacher_model.eval()  # Set to evaluation mode

# Freeze teacher model parameters
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"Teacher model loaded and frozen: {config['models']['teacher']}")

In [ ]:
# Load tokenizer (use teacher tokenizer for consistency)
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config["models"]["teacher"])

# Prepare dataset
tokenized_dataset = load_and_prepare_dataset(
    config["dataset"],
    tokenizer,
    config["tokenizer"]["chat_template"],
    config["tokenizer"]["max_length"]
)

## 6. Sequential Distillation: Smallest to Largest

Train each student model sequentially, starting from the smallest (Qwen2-0.5B).

### 6.1 Distill into Qwen2-0.5B (Smallest)

In [ ]:
# Train Qwen2-0.5B (smallest model)
student_0_5b = config["models"]["students"][0]
print(f"\nStarting distillation for: {student_0_5b}")

output_0_5b = train_student_model(
    student_model_name=student_0_5b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-0.5B distillation complete! Model saved to: {output_0_5b}")

### 6.2 Distill into Qwen2-1.5B (Medium)

In [ ]:
# Train Qwen2-1.5B (medium model)
student_1_5b = config["models"]["students"][1]
print(f"\nStarting distillation for: {student_1_5b}")

output_1_5b = train_student_model(
    student_model_name=student_1_5b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-1.5B distillation complete! Model saved to: {output_1_5b}")

### 6.3 Distill into Qwen2-7B (Largest)

In [ ]:
# Train Qwen2-7B (largest model)
student_7b = config["models"]["students"][2]
print(f"\nStarting distillation for: {student_7b}")

output_7b = train_student_model(
    student_model_name=student_7b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-7B distillation complete! Model saved to: {output_7b}")

## 7. Summary and Cleanup

In [ ]:
# Final cleanup
del teacher_model
cleanup_memory()

# Summary
print("\n" + "="*60)
print("DISTILLATION COMPLETE")
print("="*60)
print(f"\nDistilled models saved to:")
for student in config["models"]["students"]:
    model_name = student.split('/')[-1]
    output_path = os.path.join(config["training"]["output_dir"], model_name)
    print(f"  - {student}: {output_path}")

print(f"\nTeacher model: {config['models']['teacher']}")
print(f"Dataset: {config['dataset']['name']}")
print(f"Training epochs: {config['training']['num_train_epochs']}")
print(f"Distillation temperature: {config['distillation']['temperature']}")
print(f"Distillation alpha: {config['distillation']['alpha']}")

## 8. (Optional) Run All Models Sequentially

Alternatively, use this cell to run all distillations in a single loop.

In [ ]:
# Uncomment and run this cell to train all models in a loop
# (Only use if you haven't run sections 6.1-6.3)

# results = {}
# for student_model_name in config["models"]["students"]:
#     output_path = train_student_model(
#         student_model_name=student_model_name,
#         teacher_model=teacher_model,
#         tokenized_dataset=tokenized_dataset,
#         config=config
#     )
#     results[student_model_name] = output_path
#     print(f"\n{'='*40}")
#     print(f"Completed: {student_model_name}")
#     print(f"Saved to: {output_path}")
#     print(f"{'='*40}\n")

# print("\nAll distillations complete!")
# for model, path in results.items():
#     print(f"  {model}: {path}")

## 9. Model Evaluation

Evaluate and compare the performance of distilled student models against the teacher model.

Metrics computed:
- **Perplexity**: Lower is better - measures how well the model predicts the test data
- **Cross-Entropy Loss**: Lower is better - the average loss on test samples
- **KL Divergence**: Lower is better - measures how different student predictions are from teacher
- **Top-k Accuracy**: Higher is better - how often the correct token is in top-k predictions
- **Generation Quality**: Qualitative comparison of generated text

In [ ]:
import math
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader

def compute_perplexity(model, dataloader, device):
    """
    Compute perplexity on a dataset.
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing perplexity"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            
            # Count non-padded tokens
            num_tokens = attention_mask.sum().item()
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity, avg_loss


def compute_kl_divergence(student_model, teacher_model, dataloader, device, temperature=1.0):
    """
    Compute average KL divergence between student and teacher predictions.
    """
    student_model.eval()
    teacher_model.eval()
    total_kl = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing KL divergence"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            student_outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            
            student_logits, teacher_logits = pad_logits(
                student_outputs.logits, teacher_outputs.logits
            )
            
            # Apply temperature
            student_probs = F.log_softmax(student_logits / temperature, dim=-1)
            teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
            
            # Compute KL divergence
            kl_div = F.kl_div(student_probs, teacher_probs, reduction='batchmean')
            total_kl += kl_div.item()
            num_batches += 1
    
    return total_kl / num_batches


def compute_top_k_accuracy(model, dataloader, device, k_values=[1, 5, 10]):
    """
    Compute top-k accuracy for next token prediction.
    """
    model.eval()
    correct = {k: 0 for k in k_values}
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing top-k accuracy"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Shift for next token prediction
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()
            shift_mask = attention_mask[..., 1:].contiguous()
            
            # Get top-k predictions
            for k in k_values:
                top_k_preds = torch.topk(shift_logits, k, dim=-1).indices
                matches = (top_k_preds == shift_labels.unsqueeze(-1)).any(dim=-1)
                correct[k] += (matches * shift_mask.bool()).sum().item()
            
            total += shift_mask.sum().item()
    
    accuracies = {k: correct[k] / total * 100 for k in k_values}
    return accuracies


def generate_text(model, tokenizer, prompt, max_new_tokens=100, device="cuda"):
    """
    Generate text from a prompt for qualitative comparison.
    """
    model.eval()
    
    # Prepare input
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated


def evaluate_model(model, model_name, dataloader, device, teacher_model=None, temperature=2.0):
    """
    Run full evaluation suite on a model.
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    results = {"model_name": model_name}
    
    # Perplexity and Loss
    perplexity, avg_loss = compute_perplexity(model, dataloader, device)
    results["perplexity"] = perplexity
    results["avg_loss"] = avg_loss
    print(f"  Perplexity: {perplexity:.2f}")
    print(f"  Avg Loss: {avg_loss:.4f}")
    
    # Top-k Accuracy
    accuracies = compute_top_k_accuracy(model, dataloader, device)
    results["top_k_accuracy"] = accuracies
    for k, acc in accuracies.items():
        print(f"  Top-{k} Accuracy: {acc:.2f}%")
    
    # KL Divergence (only for student models)
    if teacher_model is not None:
        kl_div = compute_kl_divergence(model, teacher_model, dataloader, device, temperature)
        results["kl_divergence"] = kl_div
        print(f"  KL Divergence from Teacher: {kl_div:.4f}")
    
    return results

### 9.1 Setup Evaluation DataLoader

In [ ]:
# Evaluation configuration
eval_config = {
    "batch_size": 4,           # Batch size for evaluation
    "num_eval_samples": 500,   # Number of samples to evaluate on (None for full test set)
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# Prepare evaluation dataset
eval_dataset = tokenized_dataset["test"]
if eval_config["num_eval_samples"]:
    eval_dataset = eval_dataset.select(range(min(eval_config["num_eval_samples"], len(eval_dataset))))

# Set format for PyTorch
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Create DataLoader
eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=eval_config["batch_size"],
    shuffle=False
)

print(f"Evaluation setup:")
print(f"  Device: {eval_config['device']}")
print(f"  Batch size: {eval_config['batch_size']}")
print(f"  Eval samples: {len(eval_dataset)}")

### 9.2 Load Models for Evaluation

In [ ]:
# Load teacher model for evaluation
print("Loading teacher model for evaluation...")
teacher_model_eval = load_model(
    config["models"]["teacher"],
    use_flash_attention=config["model_config"]["use_flash_attention"]
)
teacher_model_eval = teacher_model_eval.to(eval_config["device"])
teacher_model_eval.eval()

# Load tokenizer
tokenizer_eval = AutoTokenizer.from_pretrained(config["models"]["teacher"])
tokenizer_eval.chat_template = config["tokenizer"]["chat_template"]
if tokenizer_eval.pad_token is None:
    tokenizer_eval.pad_token = tokenizer_eval.eos_token

print("Teacher model loaded for evaluation.")

### 9.3 Evaluate Teacher Model

In [ ]:
# Evaluate teacher model (baseline)
teacher_results = evaluate_model(
    model=teacher_model_eval,
    model_name=config["models"]["teacher"],
    dataloader=eval_dataloader,
    device=eval_config["device"],
    teacher_model=None  # No KL divergence for teacher
)

### 9.4 Evaluate Distilled Student Models

In [ ]:
# Evaluate all distilled student models
all_results = [teacher_results]

for student_name in config["models"]["students"]:
    model_short_name = student_name.split('/')[-1]
    model_path = os.path.join(config["training"]["output_dir"], model_short_name)
    
    # Check if model exists
    if not os.path.exists(model_path):
        print(f"\nSkipping {student_name}: Model not found at {model_path}")
        continue
    
    print(f"\nLoading distilled model from: {model_path}")
    
    # Load the distilled student model
    student_model_eval = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16
    )
    student_model_eval = student_model_eval.to(eval_config["device"])
    student_model_eval.eval()
    
    # Evaluate
    student_results = evaluate_model(
        model=student_model_eval,
        model_name=f"{student_name} (distilled)",
        dataloader=eval_dataloader,
        device=eval_config["device"],
        teacher_model=teacher_model_eval,
        temperature=config["distillation"]["temperature"]
    )
    all_results.append(student_results)
    
    # Cleanup
    del student_model_eval
    cleanup_memory()

print("\nAll models evaluated!")

### 9.5 Results Summary

In [ ]:
# Create results comparison table
import pandas as pd

def create_results_table(results_list):
    """Create a comparison table from evaluation results."""
    rows = []
    for result in results_list:
        row = {
            "Model": result["model_name"],
            "Perplexity": f"{result['perplexity']:.2f}",
            "Avg Loss": f"{result['avg_loss']:.4f}",
            "Top-1 Acc (%)": f"{result['top_k_accuracy'][1]:.2f}",
            "Top-5 Acc (%)": f"{result['top_k_accuracy'][5]:.2f}",
            "Top-10 Acc (%)": f"{result['top_k_accuracy'][10]:.2f}",
        }
        if "kl_divergence" in result:
            row["KL Div"] = f"{result['kl_divergence']:.4f}"
        else:
            row["KL Div"] = "N/A (teacher)"
        rows.append(row)
    
    return pd.DataFrame(rows)

# Display results table
print("\n" + "="*80)
print("EVALUATION RESULTS SUMMARY")
print("="*80 + "\n")

results_df = create_results_table(all_results)
print(results_df.to_string(index=False))

# Save results to CSV
results_csv_path = os.path.join(config["training"]["output_dir"], "evaluation_results.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to: {results_csv_path}")

### 9.6 Qualitative Generation Comparison

Compare generated outputs from teacher and student models on sample prompts.

In [ ]:
# Sample prompts for qualitative comparison
test_prompts = [
    "Explain the concept of machine learning in simple terms.",
    "Write a Python function to calculate the factorial of a number.",
    "What are the main differences between Python and JavaScript?",
]

def compare_generations(prompts, models_dict, tokenizer, device, max_new_tokens=150):
    """
    Generate and compare outputs from multiple models.
    """
    for prompt in prompts:
        print(f"\n{'='*80}")
        print(f"PROMPT: {prompt}")
        print("="*80)
        
        for model_name, model in models_dict.items():
            print(f"\n--- {model_name} ---")
            try:
                output = generate_text(model, tokenizer, prompt, max_new_tokens, device)
                # Extract just the assistant's response
                if "<|im_start|>assistant" in output:
                    output = output.split("<|im_start|>assistant")[-1]
                print(output.strip()[:500])  # Truncate long outputs
            except Exception as e:
                print(f"Error generating: {e}")
        print()

# Load models for generation comparison
print("Loading models for generation comparison...")
models_for_comparison = {"Teacher": teacher_model_eval}

# Load one distilled student for comparison (smallest one for speed)
student_name = config["models"]["students"][0]  # Qwen2-0.5B
model_short_name = student_name.split('/')[-1]
model_path = os.path.join(config["training"]["output_dir"], model_short_name)

if os.path.exists(model_path):
    student_model_gen = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16
    ).to(eval_config["device"])
    student_model_gen.eval()
    models_for_comparison[f"{model_short_name} (distilled)"] = student_model_gen
    print(f"Loaded distilled {model_short_name} for comparison")

# Run comparison
compare_generations(
    test_prompts,
    models_for_comparison,
    tokenizer_eval,
    eval_config["device"]
)

# Cleanup
if "student_model_gen" in dir():
    del student_model_gen
cleanup_memory()

### 9.7 Evaluation Cleanup

In [ ]:
# Final evaluation cleanup
del teacher_model_eval
cleanup_memory()

print("\nEvaluation complete!")
print(f"Results saved to: {results_csv_path}")